# 대전 아파트 매매가 예측 — 최종 제출 노트북

| 항목 | 내용 |
|---|---|
| 팀명 | (리더보드 표시 이름) |
| 팀원 | 학번 이름, 학번 이름 |
| 팀원별 기여 | 예) 홍길동: EDA·전처리 / 김데이터: 모델 튜닝 |
| 최종 평가용 제출 번호 | #( ) — 제출 이슈의 채점 댓글에 나온 "제출 번호" |
| 최종 Public 점수 | |
| 사용한 외부 데이터·사전학습 모델 | 없음 / (있으면 출처) |

**제출 규칙**: 이 노트북은 위에서 아래로 "모두 실행"했을 때 오류 없이 끝나고, 마지막 셀이 `submission.csv`를 만들어야 합니다. 리더보드에 올린 최종 제출물을 만든 코드와 같아야 하며, 난수 시드는 고정합니다. 필요 없는 셀은 지우고 섹션 제목은 유지해 주세요.


In [ ]:
# 0. 환경 설정 (수정하지 마세요)
import random, os, sys, platform
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sklearn

SEED = 42
random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.max_columns", 50)
print("python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)


## 1. 데이터 불러오기
사이트에서 직접 읽습니다. 파일을 내려받아 쓰는 경우 경로만 바꾸세요.

In [ ]:
DATA = "https://gisup89.github.io/competition/competitions/daejeon-house-price/data/"
train = pd.read_csv(DATA + "train.csv")
test = pd.read_csv(DATA + "test.csv")
print(train.shape, test.shape)
train.head()

## 2. 데이터 탐색 (EDA)
분포·왜도·상관관계·범주별 차이를 확인하고, 발견한 내용을 아래 마크다운 셀에 적습니다.

In [ ]:
train.describe().T

In [ ]:
train.hist(bins=40, figsize=(14, 9)); plt.tight_layout(); plt.show()

In [ ]:
num_cols = train.select_dtypes("number").columns.drop("id")
corr = train[num_cols].corr()["price"].sort_values(ascending=False)
print(corr)
train.groupby("gu")["price"].median().sort_values(ascending=False)

**EDA에서 발견한 것** (여기에 정리):
- 
- 

## 3. 검증 전략
학습 데이터에서 검증용을 떼어 두거나 교차검증을 사용합니다. 리더보드 Public 점수에만 맞추지 않도록 검증 점수를 함께 기록하세요.

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score

X = train.drop(columns=["id", "price"])
y = train["price"]
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=SEED)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

def rmse_cv(model, X=X, y=y):
    scores = -cross_val_score(model, X, y, scoring="neg_root_mean_squared_error", cv=cv)
    print(f"CV RMSE: {scores.mean():,.0f} ± {scores.std():,.0f}")
    return scores

## 4. 전처리 파이프라인
수치형 스케일링, 범주형(`gu`) 원-핫 인코딩, 로그 변환·파생 변수 등을 `ColumnTransformer`/`Pipeline`으로 묶습니다.

In [ ]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

num_features = ["area_m2", "floor", "built_year", "subway_dist_m", "rooms", "parking_per_hh", "brand"]
cat_features = ["gu"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
])
# TODO: 로그 변환(subway_dist_m), 파생 변수(건물 연식 = 2026 - built_year 등)를 추가해 보세요.

## 5. 모델 비교
선형회귀 → 결정트리 → 랜덤포레스트(→ 그레이디언트 부스팅) 순으로 교차검증 점수를 비교합니다.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

models = {
    "LinearRegression": LinearRegression(),
    "DecisionTree": DecisionTreeRegressor(random_state=SEED),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
}
for name, m in models.items():
    print(name, end=" | ")
    rmse_cv(Pipeline([("prep", preprocess), ("model", m)]))

## 6. 하이퍼파라미터 튜닝
`GridSearchCV` 또는 `RandomizedSearchCV`로 최종 모델을 고릅니다.

In [ ]:
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([("prep", preprocess), ("model", RandomForestRegressor(random_state=SEED, n_jobs=-1))])
param_grid = {"model__n_estimators": [300], "model__max_depth": [None, 12], "model__min_samples_leaf": [1, 3]}
search = GridSearchCV(pipe, param_grid, scoring="neg_root_mean_squared_error", cv=cv, n_jobs=-1)
search.fit(X, y)
print("best params:", search.best_params_)
print(f"best CV RMSE: {-search.best_score_:,.0f}")
final_model = search.best_estimator_

## 7. 최종 학습 · 제출 파일 생성 (이 셀의 출력 파일이 리더보드 제출물과 같아야 합니다)

In [ ]:
final_model.fit(X, y)
pred = final_model.predict(test.drop(columns=["id"]))
submission = pd.DataFrame({"id": test["id"], "price": pred})
assert len(submission) == len(test) and submission["price"].notna().all()
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("submission.csv 저장 완료 — 사이트의 '제출 파일 암호화' 페이지에서 변환해 제출하세요.")

## 8. 정리
- **접근 방법 요약**:
- **효과가 있었던 것 / 없었던 것**:
- **검증 점수와 리더보드 점수 비교**:
- **더 해 보고 싶은 것**: